In [2]:
from utils import *
import pandas as pd
import pandas as pd
import numpy as np
import os

In [2]:
# запустить
os.makedirs('ans', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('models/w2v', exist_ok=True)
os.makedirs('models/hf_towers', exist_ok=True)
os.makedirs('preprocessed', exist_ok=True)

## Препроцесс для W2V

### Полный код (нужно запустить самым первым)

1. разделение использованных пользователем фильтров и item категорий
2. нормировка текста в запросе и в кратком описании item
3. заполнение пропущенных значений в признаках item

In [3]:
# надо запустить
train = pd.read_parquet('dataset/train.parquet')
item_cols = train.columns[train.columns.str.startswith('item')]
queries_df = train.drop(columns=item_cols)
queries_df['item_id'] = train['item_id']
items_df = train[item_cols]
items_df = items_df.drop_duplicates(subset=['item_id'])

queries_df = extract_params(queries_df, 
                            column='search_infm_params_text', 
                            targets=targets, 
                            renames=queries_renames)
items_df = extract_params(items_df, 
                          column='item_infm_params_text', 
                          targets=targets, 
                          renames=items_renames)

# костыль для очистки пустых типов
x = items_df['item_vid_services']
bad_classes = list(x[x.str.startswith('место', na=False)].unique())
items_df['item_vid_services'] = items_df['item_vid_services'].replace(bad_classes, np.nan).str.strip()

x = items_df['item_type_services']
bad_classes = list(x[(x.str.startswith('место', na=False) | 
    x.str.startswith('районы', na=False) | 
    x.str.startswith('тип', na=False) | 
    x.str.startswith('куда', na=False) | 
    x.str.startswith('онлайн-показ', na=False) |
    x.str.startswith('начальная цена', na=False))].unique())
items_df['item_type_services'] = items_df['item_type_services'].replace(bad_classes, np.nan).str.strip()

queries_df = preprocess(queries_df, column='search_query')
items_df = preprocess(items_df, column='item_title_raw')

items_df['item_rating_reviews_count'] = items_df['item_rating_reviews_count'].fillna(items_df['item_rating_reviews_count'].median())
items_df['item_rating'] = items_df['item_rating'].fillna(items_df['item_rating'].mean())

In [4]:
# надо запустить
queries_df.to_parquet('preprocessed/train_queries.parquet', index=False)
items_df.to_parquet('preprocessed/train_items.parquet', index=False)

### Последовательно с кратким описанием. Запускать не нужно, суммаризация уже сверху

Сначала надо разделить табличку на items и queries, потому что дальше они иногда будут идти раздельно + хранить их так будет удобнее и места меньше заниматься (дублирование items)

In [ ]:
item_cols = train.columns[train.columns.str.startswith('item')]
queries_df = train.drop(columns=item_cols)
queries_df['item_id'] = train['item_id']
items_df = train[item_cols]
items_df = items_df.drop_duplicates(subset=['item_id'])

#### search + item filters

Выделение тегов, использованных при фильтрации

In [ ]:
targets = ["Вид услуги", "Тип услуги",]
queries_renames = {"вид услуги": "search_vid_services", "тип услуги": "search_type_services"}
items_renames = {"вид услуги": "item_vid_services", "тип услуги": "item_type_services"}

def parse_params(text, targets):
    parts = re.split(r'\s+(?=[А-Я])', text)
    idxs = {k: parts.index(k) 
            if k in parts else None 
            for k in targets }

    for k, v in idxs.items():
        if v == len(parts) - 1:
            idxs[k] = None

    result = {k.strip().lower(): parts[idx + 1].strip().lower() 
              if idx != None else None 
              for k, idx in idxs.items() }    

    return result

def extract_params(df, column, targets, renames):
    df[column] = df[column].apply(parse_params, targets=targets)
    df_expanded = pd.json_normalize(df[column]).rename(columns=renames)
    df = pd.concat([df.drop(column, axis=1), df_expanded], axis=1)
    return df


queries_df = extract_params(queries_df, 
                            column='search_infm_params_text', 
                            targets=targets, 
                            renames=queries_renames)
items_df = extract_params(items_df, 
                          column='item_infm_params_text', 
                          targets=targets, 
                          renames=items_renames)

In [ ]:
# костыль для очистки пустых типов
x = items_df['item_vid_services']
bad_classes = list(x[x.str.startswith('место', na=False)].unique())
items_df['item_vid_services'] = items_df['item_vid_services'].replace(bad_classes, np.nan).str.strip()

x = items_df['item_type_services']
bad_classes = list(x[(x.str.startswith('место', na=False) | 
    x.str.startswith('районы', na=False) | 
    x.str.startswith('тип', na=False) | 
    x.str.startswith('куда', na=False) | 
    x.str.startswith('онлайн-показ', na=False) |
    x.str.startswith('начальная цена', na=False))].unique())
items_df['item_type_services'] = items_df['item_type_services'].replace(bad_classes, np.nan).str.strip()

##### Черновик

In [ ]:
targets = [
    "Вид услуги",
    "Тип услуги",
]

def parse_params(text):
    parts = re.split(r'\s+(?=[А-Я])', text)
    idxs = {k: parts.index(k) 
            if k in parts else None 
            for k in targets }

    for k, v in idxs.items():
        if v == len(parts) - 1:
            idxs[k] = None

    result = {k.strip().lower(): parts[idx + 1].strip().lower() 
              if idx != None else None 
              for k, idx in idxs.items() }    

    return result


# idx = 0
# print(train['item_infm_params_text'][idx])
# print(parse_params(train['item_infm_params_text'][idx]))
# print(train['search_infm_params_text'][idx])
# print(parse_params(train['search_infm_params_text'][idx]))

In [ ]:
# выделение паттернов из серча
train['search_infm_params_text'] = train['search_infm_params_text'].apply(parse_params)
train_expanded = pd.json_normalize(train['search_infm_params_text']).rename(columns={"вид услуги": "search_vid_services", "тип услуги": "search_type_services"})
train = pd.concat([train.drop('search_infm_params_text', axis=1), train_expanded], axis=1)

# выделение паттернов из item
train['item_infm_params_text'] = train['item_infm_params_text'].apply(parse_params)
train_expanded = pd.json_normalize(train['item_infm_params_text']).rename(columns={"вид услуги": "item_vid_services", "тип услуги": "item_type_services"})
train = pd.concat([train.drop('item_infm_params_text', axis=1), train_expanded], axis=1)

In [ ]:
# костыль для очистки пустых типов
x = train['item_vid_services']
bad_classes = list(x[x.str.startswith('место', na=False)].unique())
train['item_vid_services'] = train['item_vid_services'].replace(bad_classes, np.nan).str.strip()

x = train['item_type_services']
bad_classes = list(x[(x.str.startswith('место', na=False) | 
    x.str.startswith('районы', na=False) | 
    x.str.startswith('тип', na=False) | 
    x.str.startswith('куда', na=False) | 
    x.str.startswith('онлайн-показ', na=False) |
    x.str.startswith('начальная цена', na=False))].unique())
train['item_type_services'] = train['item_type_services'].replace(bad_classes, np.nan).str.strip()

#### norm titles

Нормализация текста для w2v - приведение к нормальной форме + удаление артифактов

In [ ]:
# тут есть ошибка и drop, в оригинале убрал

# morph = pymorphy3.MorphAnalyzer()
# stop_words = set(stopwords.words("russian"))

# def preprocess(df, column):
#     def preprocess_(text):
#         text = str(text).lower()
#         words = re.findall(r"[a-zа-яё0-9]+", text)
#         result = []
#         for word in words:
#             if word in stop_words:
#                 continue
#             lemma = morph.parse(word)[0].normal_form
#             if lemma not in stop_words:
#                 result.append(lemma)
#         return result

#     norm_df = df[column].apply(preprocess_)
#     df[f'{column}_norm'] = norm_df
#     x = list(df[f'{column}_norm'].apply(len))
#     df = df.drop(x).reset_index(drop=True)
#     return df

# queries_df = preprocess(queries_df, column='search_query')
# items_df = preprocess(items_df, column='item_title_raw')

##### Черновик

In [ ]:
morph = pymorphy3.MorphAnalyzer()
stop_words = set(stopwords.words("russian"))

def preprocess_(text):
    text = str(text).lower()
    words = re.findall(r"[a-zа-яё0-9]+", text)
    result = []
    for word in words:
        if word in stop_words:
            continue
        lemma = morph.parse(word)[0].normal_form
        if lemma not in stop_words:
            result.append(lemma)
    return result

In [ ]:
norm_q = train['search_query'].apply(preprocess)
norm_i = train['item_title_raw'].apply(preprocess)
train['search_query_norm'] = norm_q
train['item_title_norm'] = norm_i

In [ ]:
x = train['search_query_norm'].apply(len)
y = train['item_title_norm'].apply(len)
drop_indx = list(set(y[y == 0].index).union(set(x[x == 0].index)))

In [ ]:
train = train.drop(drop_indx).reset_index(drop=True)

#### fill na vals

Заполнение пустых значений рейтингов медианным и средним значением

In [ ]:
items_df['item_rating_reviews_count'] = items_df['item_rating_reviews_count'].fillna(items_df['item_rating_reviews_count'].median())
items_df['item_rating'] = items_df['item_rating'].fillna(items_df['item_rating'].mean())

##### Черновик

In [ ]:
# Втупую
train['item_rating_reviews_count'] = train['item_rating_reviews_count'].fillna(train['item_rating_reviews_count'].median())
train['item_rating'] = train['item_rating'].fillna(train['item_rating'].mean())

#### save

In [ ]:
queries_df.to_parquet('dataset/train_queries.parquet', index=False)
items_df.to_parquet('dataset/train_items.parquet', index=False)

##### Черновик

In [ ]:
train.to_parquet('dataset/train_preprocces.parquet', index=False)

In [ ]:
item_cols = train.columns[train.columns.str.startswith('item')]
queries_df = train.drop(columns=item_cols)
queries_df['item_id'] = train['item_id']
items_df = train[item_cols]
items_df = items_df.drop_duplicates(subset=['item_id'])

In [ ]:
queries_df.to_parquet('dataset/train_queries.parquet', index=False)
items_df.to_parquet('dataset/train_items.parquet', index=False)

## Препроцесс тестовой выборки для W2V

In [5]:
# запустить после препроцесса

queries_df_test = pd.read_parquet('dataset/benchmark_queries.parquet')
items_df_test = pd.read_parquet('dataset/benchmark_items.parquet')

targets = ["Вид услуги", "Тип услуги",]
queries_renames = {"вид услуги": "search_vid_services", "тип услуги": "search_type_services"}
items_renames = {"вид услуги": "item_vid_services", "тип услуги": "item_type_services"}

queries_df_test = extract_params(queries_df_test, 
                            column='search_infm_params_text', 
                            targets=targets, 
                            renames=queries_renames)
items_df_test = extract_params(items_df_test, 
                          column='item_infm_params_text', 
                          targets=targets, 
                          renames=items_renames)
print('Экстракция')

x = items_df_test['item_vid_services']
bad_classes = list(x[x.str.startswith('место', na=False)].unique())
items_df_test['item_vid_services'] = items_df_test['item_vid_services'].replace(bad_classes, np.nan).str.strip()
x = items_df_test['item_type_services']
bad_classes = list(x[(x.str.startswith('место', na=False) | 
    x.str.startswith('районы', na=False) | 
    x.str.startswith('тип', na=False) | 
    x.str.startswith('куда', na=False) | 
    x.str.startswith('онлайн-показ', na=False) |
    x.str.startswith('начальная цена', na=False))].unique())
items_df_test['item_type_services'] = items_df_test['item_type_services'].replace(bad_classes, np.nan).str.strip()
print('Костыль')

queries_df_test = preprocess(queries_df_test, column='search_query')
items_df_test = preprocess(items_df_test, column='item_title_raw')
print('Нормализация')

items_df_test['item_rating_reviews_count'] = items_df_test['item_rating_reviews_count'].fillna(items_df_test['item_rating_reviews_count'].median())
items_df_test['item_rating'] = items_df_test['item_rating'].fillna(items_df_test['item_rating'].mean())

Экстракция
Костыль
Нормализация


In [6]:
queries_df_test.to_parquet('preprocessed/benchmark_queries_preprocessed_w2v.parquet', index=False)
items_df_test.to_parquet('preprocessed/benchmark_items_preprocessed_w2v.parquet', index=False)

## Препроцесс для HF моделей

Перепроцессим train_dataset. Нам его нужно только немного очистить. Можем использовать данные, которые уже обработали

In [ ]:
# после препроцесса w2v запустить
queries_df_hf = pd.read_parquet('preprocessed/train_queries.parquet')
items_df_hf = pd.read_parquet('preprocessed/train_items.parquet')
queries_df_hf['search_query_norm'] = queries_df_hf['search_query'].apply(preprocess_hf, prefix='query: ')
items_df_hf['item_title_raw_norm'] = items_df_hf['item_title_raw'].apply(preprocess_hf, prefix='passage: ')

In [6]:
# запустить
queries_df_hf.to_parquet('preprocessed/train_queries_hf.parquet', index=False)
items_df_hf.to_parquet('preprocessed/train_items_hf.parquet', index=False)

также обработаем описание item для гибридной модели (w2v + transformer)

In [21]:
items_df_hf = pd.read_parquet('preprocessed/train_items.parquet')
items_df_hf['item_description_raw_norm'] = items_df_hf['item_description_raw'].apply(preprocess_hf, prefix='passage: ')

In [ ]:
items_df_hf.to_parquet('preprocessed/train_items_hybrid.parquet', index=False)

## Препроцесс тестовой выборки для HF

In [ ]:
queries_df_test = pd.read_parquet('preprocessed/benchmark_queries_preprocessed_w2v.parquet')
items_df_test = pd.read_parquet('preprocessed/benchmark_items_preprocessed_w2v.parquet')
queries_df_test['search_query_norm'] = queries_df_test['search_query'].apply(preprocess_hf, prefix='query: ')
items_df_test['item_title_raw_norm'] = items_df_test['item_title_raw'].apply(preprocess_hf, prefix='passage: ')

In [10]:
queries_df_test.to_parquet('preprocessed/benchmark_queries_preprocessed_hf.parquet', index=False)
items_df_test.to_parquet('preprocessed/benchmark_items_preprocessed_hf.parquet', index=False)

гибрид

In [2]:
items_df_hf = pd.read_parquet('preprocessed/benchmark_items_preprocessed_w2v.parquet')
items_df_hf['item_description_raw_norm'] = items_df_hf['item_description_raw'].apply(preprocess_hf, prefix='passage: ')

In [ ]:
items_df_hf.to_parquet('preprocessed/benchmark_items_preprocessed_hybrid.parquet', index=False)